# 03 - Analyse non supervisée & labellisation faible
**Branche** : `feature/03-clustering`

PCA → K-Means / GMM / Agglomératif / DBSCAN (k = 2 comme le jeu labellisé) ; ARI calculé sur les 100 images labellisées. **Ne pas attendre un ARI parfait.**

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from brainscanai.config import load_config, resolve_path
from brainscanai.utils import set_seed

cfg = load_config()
set_seed(cfg["seed"])
PROC, FEAT = resolve_path(cfg, "processed", create=True), resolve_path(cfg, "features", create=True)
FIG, TAB = resolve_path(cfg, "figures", create=True), resolve_path(cfg, "tables", create=True)
from brainscanai.features.io import load_features
from brainscanai.clustering.reduce import *
from brainscanai.clustering.cluster import run_all
from brainscanai.clustering.evaluate import evaluate_all
from brainscanai.evaluation.plots import plot_embedding_2d, save_fig

In [ ]:
cc = cfg['clustering']
manifest = pd.read_csv(PROC / 'manifest_clean.csv')
X, m = load_features(FEAT / 'embeddings.npz', manifest)
Xs, _ = standardize(X)
Z, pca = pca_reduce(Xs, cc['pca_components'], cfg['seed'])
Z.shape

## Visualisation 2D (t-SNE sur la PCA) - vérité terrain

In [ ]:
coords = tsne_embed(Z, cc['tsne_perplexity'], cfg['seed'])
plot_embedding_2d(coords, m.label_name, 'Classes connues (labellisées) vs inconnues'); plt.show()

## Comparaison de 4 algorithmes

In [ ]:
results = run_all(Z, cc['n_clusters'], cfg['seed'], cc['dbscan_min_samples'])
table = evaluate_all(results, Z, m.label.to_numpy())
table.to_csv(TAB / 'clustering_comparison.csv')
table

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (name, r) in zip(axes, results.items()):
    plot_embedding_2d(coords, r.labels, f"{name} (ARI={table.loc[name, 'ARI']:.2f})", ax)
save_fig(fig, 'clustering_tsne', FIG); plt.show()

## Sensibilité : k et dimension de la PCA

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
mask = m.label.to_numpy() >= 0
rows = []
for k in [2, 3, 4, 5]:
    lab = KMeans(k, n_init=20, random_state=cfg['seed']).fit_predict(Z)
    rows.append(dict(k=k, silhouette=silhouette_score(Z, lab), ARI=adjusted_rand_score(m.label[mask], lab[mask])))
pd.DataFrame(rows).round(3)

## Interprétation
_À compléter : quelle méthode est la meilleure (ARI, pureté, silhouette) ? Les clusters suivent-ils la pathologie ou plutôt l'intensité/la coupe/le scanner ? Regarder des images de chaque cluster._

In [ ]:
from brainscanai.evaluation.plots import plot_sample_grid
best = results[cc['selected_method']]
for c in sorted(set(best.labels[best.labels >= 0])):
    idx = np.where(best.labels == c)[0][:5]
    plot_sample_grid(m.path.iloc[idx].tolist(), [f'cluster {c}'] * len(idx)); plt.show()

## Labellisation faible
Le mapping cluster → classe est appris sur `strong_train` **uniquement** (voir notebook 04, où le split est créé). Les labels faibles restent dans un jeu séparé (`weak_pool`), jamais mélangé aux labels forts.